# M5.1 · OpenVINO CPU text encoder — the recovery and the fair A/B

**Learning objective:** validate the OpenVINO BF16 CPU encoder recovery (Part A: local encoder speed), the real end-to-end architecture benefit (Part B: GPU pipeline with CPU TE offload), and the **fair factorial** that separates architecture effects from client-monitoring effects (Part C). Encoder speedup ≠ image throughput gain — never conflate them.

Reference lesson: [M5.1 lesson](../course/M05_parallel_heterogeneous/M5_1_OPENVINO_CPU_TE.md) · [中文](./06_openvino_and_fair_ab.zh-CN.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse the latest physical run (shown run_id)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="06")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse the latest physical run to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing the latest physical campaign run(s) "
          "instead of launching new ones (metered-GPU mode)")
    return False

## 0 · Artifact admission — PASS vs PASS_WITH_WARNINGS
The OV IRs are verified byte-level against their conversion manifests (streaming SHA256, source-model digest chain). An unpinned field keeps the verdict at `PASS_WITH_WARNINGS` — that distinction is preserved, never rounded up to PASS.

In [ ]:
def ov_verify():
    import subprocess, sys
    r = subprocess.run([sys.executable, "-c",
        "import os,sys; sys.path.insert(0,'.');"
        "from lab.m51_contract import verify_artifact;"
        "root=os.environ.get('M51_OV_ARTIFACTS', 'runs/m51-ov-artifacts');"
        "print(verify_artifact(root+'/bf16',"
        "['language_model_bf16.xml','language_model_bf16.bin'],"
        "expected_identity={'storage':'FP16','runtime_precision':'bf16'}))"],
        cwd=str(ROOT), capture_output=True, text=True, timeout=300)
    print(r.stdout or r.stderr)
    return r.returncode

rc = gpu_cell(ov_verify)
if rc is None:
    print("REFERENCE mode: artifact verification runs on the measurement host; "
          "campaign status recorded bf16 = PASS_WITH_WARNINGS (xml digest "
          "computed but not pinned by the historical manifest)")

## Part A · Local encoder: GPU PyTorch vs CPU PyTorch vs OpenVINO BF16
Frozen reference ratios (encoder only, same CPU): OV BF16 is ~331–480× faster than PyTorch CPU. **Do not blindly assert your host reproduces this** — check the campaign/current records below.

In [ ]:
import json
enc = ns.read_json("runs/m51-20261105-cleanroom-01/performance/encoder_ab.json")
for arm in ("G_PyTorch_GPU", "C0_PyTorch_CPU", "C1b_OV_BF16"):
    row = enc["arms"][arm]["prompts"]["p1-zh-scene"]
    print(f"{arm:16s} warm median {row['warm_median_s']:8.3f}s  n={row['n_warm']}")
_stamp(HISTORICAL, "frozen encoder A/B (cleanroom campaign, 16-thread cgroup quota)")

def latest_h3():
    cands = sorted((ROOT / "runs").glob("*H3_OV_BF16/H3_RESULTS.json"))
    if not cands:
        return None, None
    return json.loads(cands[-1].read_text()), cands[-1].parent.name

h3, h3_id = gpu_cell(latest_h3) or (None, None)
if h3 is not None:
    session.note_consumed(h3_id, why="current-campaign OV encoder record")
    print("gates:", json.dumps(h3.get("gates", {}), indent=1))
    _stamp(MEASURED_TODAY, f"current-campaign H3_OV_BF16 gates above ({h3.get('mode')})")
    print("release_status:", h3.get("release_status"))
else:
    print("no current-campaign OV encoder record on this host; Part A stays "
          "reference-scoped here")

## Part B · End-to-end: GPU-native (A) vs OpenVINO-CPU TE (C)
Same GPU, DiT, VAE, scheduler, resolution, prompts, seeds and client policy; four real requests per arm. The honest metric is **batch wall time / images per minute**, with service latency reported separately.

In [ ]:
def latest_pipeline_ab():
    cands = sorted((ROOT / "runs").glob("*m51pipe*/pipeline_ab.json")) + \
            sorted((ROOT / "runs").glob("m51-*/pipeline/pipeline_ab.json"))
    if not cands:
        return None
    import json as _j
    return _j.loads(cands[-1].read_text())

pab = gpu_cell(latest_pipeline_ab)
if pab is not None:
    for arm, row in pab["arms"].items():
        if isinstance(row, dict) and "batch_wall_s" in json.dumps(row)[:400]:
            print(arm, {k: row.get(k) for k in ("batch_wall_s", "images_per_min",
                                                "gpu_total_s", "cpu_encode_s")
                        if k in row})
    _stamp(MEASURED_TODAY, "pipeline A/C arms from the current campaign")
else:
    frozen = ns.read_json("runs/m51-20261105-cleanroom-01/pipeline/pipeline_ab.json")
    for arm in frozen["arms"]:
        print("frozen", arm, str(frozen["arms"][arm])[:160])
    _stamp(HISTORICAL, "frozen cleanroom pipeline A/B")

## Part C · The fair factorial — architecture × client monitoring
Comparing C-fast against A-original overstates the architecture effect: A-original also pays per-request monitor teardown. The 2×2 factorial (A/C × original/fast) isolates both axes. The old study found a *small* architecture benefit under matched clients and a larger client effect — treat as hypotheses.

In [ ]:
ff = ns.fair_factorial(clean_head=False)
cells = ff["cells"]
print(f"{'cell':12s} {'n':>2s} {'median_s':>9s} {'img/min':>8s}")
for k, v in cells.items():
    print(f"{k:12s} {v['n']:2d} {v['median_s']:9.2f} {v['images_per_min']:8.2f}")
arch = cells["A_original"]["median_s"] / cells["C_original"]["median_s"]
arch_fair = cells["A_fast"]["median_s"] / cells["C_fast"]["median_s"]
client_a = cells["A_original"]["median_s"] / cells["A_fast"]["median_s"]
client_c = cells["C_original"]["median_s"] / cells["C_fast"]["median_s"]
_stamp(HISTORICAL,
       f"fair factorial: architecture effect {arch:.3f}x (original clients) "
       f"vs {arch_fair:.3f}x (matched fast clients); client effect "
       f"A {client_a:.3f}x / C {client_c:.3f}x")
print("never report C_fast vs A_original as a pure architecture effect")

## Correctness scope — what stays FAIL and what stays human
1. The INT8 encoder arm's numerical **FAIL stays FAIL** — no automatic quality promotion anywhere in this course.
2. The C1b owner acceptance is scoped to the **historically evaluated deployment**; your new physical run needs its own correctness evidence.

> **Quality decisions stay human.** No cell auto-approves an image-quality verdict. Historical owner verdicts apply **only** to their originally evaluated deployment scope; a new physical run needs its own review (`PENDING_HUMAN_REVIEW` until then).

## Your turn
1. Compute encoder-only speedup vs pipeline-level speedup from the cells above — why do they differ by an order of magnitude?
2. Which Part-C cell would you double-sample to tighten the architecture estimate, and why?
3. Propose the RAM admission you would run before a PyTorch CPU full arm on a 55 GiB cgroup.

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## Acceptance criteria

- [ ] You can state which numbers are **measured on this GPU today**, which are
      **historical reference values**, and which are **estimates/simulations**.
- [ ] Every physical run you produced has a visible `run_id` under `runs/`.
- [ ] You did not present a smoke/classroom population as a formal result.
- [ ] You can explain one limitation of each measurement you reported.

## Reproduce & contribute

```bash
python -m lab doctor                                  # admission evidence -> runs/doctor.json
python -m lab run --experiment H3_OV_BF16 --date <your-tag>  # full research protocol
python -m lab report --scope student --kind all       # aggregate YOUR runs only
```

Commit your `runs/<run-id>/*.json` records (never `evidence/`) and open a PR
describing: hypothesis → protocol → numbers → limitations. Negative results
are welcome — a measured regression is a scientific outcome, not a failure.